# Qwen2.5-7B LoRA (Kaggle Lite)

Notebook nay duoc thiet ke cho GPU nhe hon tren Kaggle (T4/P100), uu tien chay on dinh truoc.

Luong chay:
1. Kiem tra GPU
2. Clone/cap nhat repo
3. Cai dependencies
4. Tao config lite tu config goc
5. Train LoRA
6. Xem log va metrics

In [ ]:
import os
import subprocess
from pathlib import Path


def run(cmd: str):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

run("nvidia-smi")

In [ ]:
REPO_URL = "https://github.com/pluto0203/Qwen2.5B_finetune.git"
REPO_DIR = Path("/kaggle/working/Qwen2.5B_finetune")

if not REPO_DIR.exists():
    run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    run(f"cd {REPO_DIR} && git pull")

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())

run("python -m pip install -U pip")
run("python -m pip install -r requirements.txt")

In [ ]:
import yaml

base_cfg_path = REPO_DIR / "configs" / "train_qwen25_7b_kaggle_lite.yaml"
run_cfg_path = REPO_DIR / "configs" / "train_qwen25_7b_kaggle_lite_runtime.yaml"

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Lite profile for Kaggle T4/P100
cfg["model"]["load_in_4bit"] = True
cfg["model"]["attn_implementation"] = "sdpa"

cfg["data"]["max_seq_length"] = 768
cfg["data"]["packing"] = False
# Set >0 for faster smoke-test, e.g. 5000
cfg["data"].setdefault("limit_rows", None)

cfg["lora"]["r"] = 8
cfg["lora"]["alpha"] = 16
cfg["lora"]["dropout"] = 0.05

cfg["training"]["output_dir"] = "/kaggle/working/outputs/qwen25_7b_medqa_lora_kaggle_lite"
cfg["training"]["per_device_train_batch_size"] = 1
cfg["training"]["per_device_eval_batch_size"] = 1
cfg["training"]["gradient_accumulation_steps"] = 8
cfg["training"]["num_train_epochs"] = 1
cfg["training"]["learning_rate"] = 2e-4
cfg["training"]["gradient_checkpointing"] = True
cfg["training"]["optim"] = "paged_adamw_32bit"
cfg["training"]["bf16"] = False
cfg["training"]["fp16"] = True
cfg["training"]["dataloader_num_workers"] = 2
cfg["training"]["dataloader_persistent_workers"] = False
cfg["training"]["logging_steps"] = 20
cfg["training"]["eval_steps"] = 200
cfg["training"]["save_steps"] = 200

with open(run_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Saved runtime config:", run_cfg_path)
print(yaml.safe_dump(cfg, sort_keys=False)[:2500])

In [ ]:
run(f"python -m src.train_lora --config {run_cfg_path}")

In [ ]:
import pandas as pd

out_dir = Path("/kaggle/working/outputs/qwen25_7b_medqa_lora_kaggle_lite")
logs_path = out_dir / "logs" / "metrics.jsonl"
metrics_yaml = out_dir / "eval_metrics.yaml"

print("Output dir exists:", out_dir.exists())
print("Metrics jsonl:", logs_path)
print("Eval metrics:", metrics_yaml)

if logs_path.exists():
    df = pd.read_json(logs_path, lines=True)
    display(df.tail(20))
else:
    print("metrics.jsonl not found yet")